# 世界通史时空图谱 · AI 图生视频工坊（LTX-Video · Colab L4 / A100）

使用开源免鉴权图生视频模型 **`Lightricks/LTX-Video`**，直接以站点现有静态历史油画作首帧生成 **4–5 秒平滑循环动态史画短视频（MP4）**。

生成结束后打包为 `atlas-videos-pack.zip`，回到本地运行：
```powershell
npm.cmd run media:import -- C:\Users\lq\Downloads\atlas-videos-pack.zip --type videos
```
即可自动激活前端右侧悬浮框与全屏弹层的原生 `<video>` 高清动态播放！

In [ ]:
# Cell 1: 检查 GPU 算力（建议选择 L4 22GB 或 A100 40GB；完成后请及时断开释放）
!nvidia-smi
import torch
assert torch.cuda.is_available(), '请先在 Runtime -> Change runtime type 中选择 L4 或 A100 GPU！'
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'✅ 检测到 GPU: {torch.cuda.get_device_name(0)} ({vram_gb:.1f} GB VRAM)')

In [ ]:
# Cell 2: 安装 Diffusers 与 FFmpeg 视频编码依赖
!pip install -q -U "diffusers>=0.32.0" "transformers>=4.46.0" accelerate sentencepiece "imageio[ffmpeg]" pillow requests

In [ ]:
# Cell 3: 拉取线上历史插画清单与原画 WebP
import os, json, requests

SITE_MEDIA_BASE = 'https://world-history-atlas-ckn.pages.dev/media/illustrations'
SRC_DIR = '/content/src_illustrations'
os.makedirs(SRC_DIR, exist_ok=True)

manifest = requests.get(f'{SITE_MEDIA_BASE}/manifest.json', timeout=30).json()
items = manifest.get('items', {})
print(f'📚 已载入插画清单：共 {len(items)} 项（model={manifest.get("model")}）')

FLAGSHIP_KEYS = [
    'era:era-1', 'era:era-2', 'era:era-3', 'era:era-4',
    'era:era-5', 'era:era-6', 'era:era-7', 'era:era-8',
    'event:e1-pyramid', 'event:e2-qin-unification', 'event:e2-roman-empire',
    'event:e2-christianity', 'event:e3-tang-chang-an', 'event:e4-song-commerce',
    'event:e4-mongol-empire', 'event:e4-gutenberg', 'event:e5-columbus',
    'event:e5-voc', 'event:e6-watt-steam', 'event:e6-second-industrial',
    'event:e7-ww2', 'event:e7-apollo-arpanet', 'event:e8-china-wto', 'event:e8-generative-ai',
]
print('⭐ 默认 flagship 批次条目数:', len(FLAGSHIP_KEYS))

In [ ]:
# Cell 4: 使用 Lightricks/LTX-Video 批量图生视频
import datetime, time
import numpy as np
from PIL import Image
import imageio
from diffusers import LTXImageToVideoPipeline

BATCH_MODE = 'flagship'  # 'flagship' (24项核心) | 'all_eras' (8个时代) | 'custom'
CUSTOM_KEYS = []         # 当 BATCH_MODE='custom' 时填写如 ['event:e5-columbus']
NUM_FRAMES = 97          # ~4 秒 @ 24fps (需满足 8*k + 1)
WIDTH, HEIGHT = 704, 384 # 16:9 且可被 32 整除
NUM_INFERENCE_STEPS = 28
SEED = 42

OUT_DIR = '/content/atlas_videos_out'
os.makedirs(OUT_DIR, exist_ok=True)

if BATCH_MODE == 'all_eras':
    target_keys = [k for k in items.keys() if k.startswith('era:')]
elif BATCH_MODE == 'custom' and CUSTOM_KEYS:
    target_keys = [k for k in CUSTOM_KEYS if k in items]
else:
    target_keys = [k for k in FLAGSHIP_KEYS if k in items]

print(f'🎬 本次计划生成 {len(target_keys)} 条动态史画视频 ({WIDTH}x{HEIGHT}, {NUM_FRAMES} 帧)')

pipe = LTXImageToVideoPipeline.from_pretrained(
    'Lightricks/LTX-Video',
    torch_dtype=torch.bfloat16,
).to('cuda')
pipe.enable_vae_tiling()

out_manifest = {
    'version': 1,
    'generatedAt': datetime.datetime.utcnow().isoformat() + 'Z',
    'model': 'Lightricks/LTX-Video',
    'items': {},
}

t0 = time.time()
for idx, key in enumerate(target_keys, 1):
    entry = items[key]
    rel_img = entry['file']
    img_url = f'{SITE_MEDIA_BASE}/{rel_img}'
    local_img = os.path.join(SRC_DIR, rel_img.replace('/', '_'))
    if not os.path.exists(local_img):
        r = requests.get(img_url, timeout=30)
        r.raise_for_status()
        with open(local_img, 'wb') as f:
            f.write(r.content)

    init_image = Image.open(local_img).convert('RGB').resize((WIDTH, HEIGHT), Image.Resampling.LANCZOS)
    rel_mp4 = os.path.splitext(rel_img)[0] + '.mp4'
    dst_mp4 = os.path.join(OUT_DIR, rel_mp4)
    os.makedirs(os.path.dirname(dst_mp4), exist_ok=True)

    caption = entry.get('caption') or key
    motion_prompt = (
        f'Subtle cinematic documentary camera slow push-in on a classical historical museum oil painting: {caption}. '
        'Gentle drifting sunlight rays, soft moving clouds, shimmering water, billowing banners and cloth folds, '
        'painterly brushwork preserved, smooth 24fps museum film lighting, no jarring cuts.'
    )

    generator = torch.Generator(device='cuda').manual_seed(SEED + idx)
    video_frames = pipe(
        image=init_image,
        prompt=motion_prompt,
        negative_prompt='worst quality, inconsistent motion, blurry, jittery, distorted faces, text watermark',
        width=WIDTH,
        height=HEIGHT,
        num_frames=NUM_FRAMES,
        num_inference_steps=NUM_INFERENCE_STEPS,
        generator=generator,
    ).frames[0]

    with imageio.get_writer(dst_mp4, fps=24, codec='libx264', quality=7, pixelformat='yuv420p') as writer:
        for fr in video_frames:
            writer.append_data(np.array(fr))

    out_manifest['items'][key] = {
        'file': rel_mp4,
        'durationSec': round(NUM_FRAMES / 24.0, 2),
        'width': WIDTH,
        'height': HEIGHT,
        'model': 'Lightricks/LTX-Video',
        'caption': caption,
    }
    mb = os.path.getsize(dst_mp4) / (1024 * 1024)
    print(f'[{idx}/{len(target_keys)}] ✅ {key} -> {rel_mp4} ({mb:.2f} MB) | 耗时 {time.time() - t0:.1f}s')

with open(os.path.join(OUT_DIR, 'manifest.json'), 'w', encoding='utf-8') as f:
    json.dump(out_manifest, f, ensure_ascii=False, indent=2)

print('🎉 视频批处理完成！')

In [ ]:
# Cell 5: 打包下载 atlas-videos-pack.zip
import shutil
from google.colab import files

ZIP_PATH = '/content/atlas-videos-pack'
archive = shutil.make_archive(ZIP_PATH, 'zip', OUT_DIR)
size_mb = os.path.getsize(archive) / (1024 * 1024)
print(f'📦 已打包 {len(out_manifest["items"])} 段动态史画视频 -> {archive} ({size_mb:.2f} MB)')
files.download(archive)